# 夸父金融 D3：客户端契约与位序溯源

本附件只读回放 13 项真实实验的已有证据；不读取密钥、不导入或创建 Task、不查询云、不提交电路。
当前是 13/20 项，保留 7 项；金融数值验收仍 INCONCLUSIVE。
本轮确认软件版本和项目侧转换链，不把某份文档的字段补写进旧任务响应。


In [1]:
import json, hashlib
from pathlib import Path
import pandas as pd
from IPython.display import display
root = Path.cwd()
while not (root / "quantumbridge/finance").is_dir() and root != root.parent:
    root = root.parent
base = root / "output/finance-hardware/diagnostics/d3"
load = lambda name: json.loads((base / name).read_text())
audit = load("d3_no_submit_audit.json")
assert audit["total_tasks"] == 13 and audit["new_submissions"] == 0
assert all(hashlib.sha256((root / name).read_bytes()).hexdigest() == digest
           for name, digest in audit["file_hashes"].items())
print("全部原始证据散列未变，任务13/20，新增0，保留7。")


全部原始证据散列未变，任务13/20，新增0，保留7。


## 1. 当前安装的是什么

`quark` 命名空间同时出现于多个发行包的元数据中；以实际 Task 所在文件与发行包文件清单对应，
确认 `quark.quafu.Task` 来自 `quarkstudio 7.3.9`。这里只检查公开签名、帮助和安装元数据，
不读取上游实现或测试源码，不实例化需凭据的客户端。
另一个 `quafu-sdk` 发行包未安装，不能拿其 `Result.bit_order` 当旧服务响应的事实。


In [2]:
contract = load("installed_client_contract.json")
display(pd.DataFrame([{"distribution": name, **value} for name, value in contract["distributions"].items()]))
print("Python:", contract["python_executable"])
print("Actual Task:", contract["Task_module"], contract["Task_module_path"])
display(contract["Task_public_method_signatures"])
assert "repeat" in contract["Task_public_method_signatures"]["run"]
assert "compile" in contract["gateway_signatures"]["submit"]
assert not contract["Task_instantiated"]
display(contract["call_path"])


,distribution,installed,version,lists_live_Task_module
0,quarkstudio,True,7.3.9,[quark/quafu.py]
1,quarkcircuit,True,0.5.13,[]
2,pyquafu,True,0.4.5,[]
3,quafu-sdk,False,NaN,NaN


Python: /Users/avalok/work/showtime/.venv/bin/python
Actual Task: quark.quafu /Users/avalok/work/showtime/.venv/lib/python3.12/site-packages/quark/quafu.py


{'backend': "(self, chip: str, show_couplers_fidelity: bool = False, show_quibts_attributes: Literal['T1', 'T2', 'fidelity', 'frequancy', ''] = '', highlight_nodes: list = [], save_svg_fname: str | None = None)",
 'cancel': '(self, tid: int)',
 'delete': '(self, tid: int)',
 'query': "(self, tid: int = 2, chips: str = 'Baihua', status: str = 'Finished,Failed', start: str = '2024-04-01-01-23-45', end: str = '2026-10-23-13-43-13', offset: int = 0, limit: int = 10, sort: Literal['taskId', 'taskName', 'chipName', 'status', 'submitTime'] = 'submitTime', order: Literal['asc', 'desc'] = 'desc')",
 'request': "(self, url: str, data: dict = {}, method: str = 'get')",
 'result': '(self, tid: int, timeout: float = 0.0)',
 'run': '(self, task: dict, repeat: int = 1)',
 'status': '(self, tid: int = 0)',
 'verify': '(self)'}

['FinanceCloudSession._raw_job',
 'QuafuCloudBackend.submit',
 'QuarkStudioTransport.submit',
 'quark.quafu.Task.run(task, repeat=shots//1024)',
 'QuafuCloudJob.refresh',
 'QuarkStudioTransport.retrieve',
 'quark.quafu.Task.result(task_id)',
 'gateway _extract_counts',
 'finance explicit normalization']

## 2. 能力矩阵：文档、签名和实际观测分开

旧 QuarkStudio 文档展示 `options.target_qubits`；这不证明本机 gateway 接受它，
更不证明 Dongling 一定保持指定物理集合或有序映射。
`compile=False` 仍属于提交执行的任务参数，不等于免费 compile-only 或 dry-run。
实际 13 份响应没有 bit_order、compiler version 或 calibration ID；不回填、不猜测。


In [3]:
display(pd.DataFrame([{"capability": key, **value} for key, value in load("public_api_capability_matrix.json").items()]))
fields = load("historical_response_field_matrix.json")
display(pd.DataFrame([{"experiment": key, "bit_order": value["bit_order"]["status"],
    "compiler": value["compiler"]["status"], "calibration_id": value["calibration_id"]["status"],
    "transpiled": value["transpiled"]["status"]} for key, value in fields.items()]))
assert all(value["bit_order"]["observed_value"] is None for value in fields.values())


,capability,status,evidence,scope,platform_status,adapter_status
0,read_existing_task,VERIFIED_READ_ONLY,thirteen previously saved Task.result response...,NaN,NaN,NaN
1,result_metadata,VERIFIED_READ_ONLY,actual response fields audited per receipt,NaN,NaN,NaN
2,raw_bit_order,ABSENT,no bit_order declaration in any of thirteen re...,NaN,NaN,NaN
3,physical_measurement_mapping,VERIFIED_READ_ONLY,"native QASM/QLISP terminal measurement maps, n...",NaN,NaN,NaN
4,compiler_version,ABSENT,no historical compiler version in receipts; pa...,NaN,NaN,NaN
5,calibration_id,ABSENT,no historical calibration ID in receipts,NaN,NaN,NaN
6,compile_only,UNSUPPORTED,no compile-only public Task method found; comp...,current project adapter,DOCUMENTED_UNVERIFIED,NaN
7,physical_layout_selection,DOCUMENTED_UNVERIFIED,historical options.target_qubits example is no...,NaN,NaN,UNSUPPORTED
8,dry_run,UNSUPPORTED,no documented zero-side-effect method verified...,current project adapter,DOCUMENTED_UNVERIFIED,NaN
9,target_availability,DOCUMENTED_UNVERIFIED,public Task.status/backend signatures exist; n...,NaN,NaN,NaN


,experiment,bit_order,compiler,calibration_id,transpiled
0,calibration/bit-order,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
1,allocation/qaoa,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
2,risk/base,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
3,risk/stress,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
4,derivatives/risk_neutral_paths,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
5,constraints/qaoa,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
6,correlation/positive_correlation,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
7,correlation/negative_correlation,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
8,correlation/time_sampling,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY
9,diagnostic/3q-zero,ABSENT,ABSENT,ABSENT,VERIFIED_READ_ONLY


## 3. Counts 逐层转换：哪里已经证明，哪里没有

保存的原始证据是 **Task.result 已解码的 Python 响应**，不是原始 HTTP 字节；
因此不能声称已证明服务端到 SDK 内部完全没有换序。项目 gateway 的 `_extract_counts` 不改变标签；
8 项业务结果由显式 q0-last 假设执行反转，所有转换可严格逆向恢复，shots 不丢失也不重复。
这个假设状态标成 **GATEWAY_ASSUMPTION_UNVERIFIED**，不是已知服务端端序。
原 2q 校准和 D1 四项不是金融解码输入，单独标成 NOT_USED_FOR_FINANCE_DECODING。


In [4]:
traces = load("raw_label_provenance.json")
display(pd.DataFrame([{"experiment": key, "width": value["label_width"], "shots": value["shots"],
    "client_decode": value["client_decoding_stage"]["status"],
    "gateway_changes_labels": value["project_extraction_changes_labels"],
    "financial_order": value["saved_finance_decoder_order"], "status": value["finance_decoder_status"],
    "reversible": value["project_transforms_exactly_invertible"]} for key, value in traces.items()]))
assert all(value["project_transforms_exactly_invertible"] for value in traces.values())
example = traces["derivatives/risk_neutral_paths"]
display(example["stages"])
display(example["hypotheses"])


,experiment,width,shots,client_decode,gateway_changes_labels,financial_order,status,reversible
0,calibration/bit-order,2,1024,DOCUMENTED_UNVERIFIED,False,None,NOT_USED_FOR_FINANCE_DECODING,True
1,allocation/qaoa,4,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
2,risk/base,3,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
3,risk/stress,3,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
4,derivatives/risk_neutral_paths,3,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
5,constraints/qaoa,4,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
6,correlation/positive_correlation,4,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
7,correlation/negative_correlation,4,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
8,correlation/time_sampling,3,1024,DOCUMENTED_UNVERIFIED,False,q0-last,GATEWAY_ASSUMPTION_UNVERIFIED,True
9,diagnostic/3q-zero,3,1024,DOCUMENTED_UNVERIFIED,False,None,NOT_USED_FOR_FINANCE_DECODING,True


['server raw HTTP labels UNOBSERVED',
 'installed Task.result decoding UNVERIFIED',
 'saved client response count',
 'gateway _extract_counts: labels unchanged',
 'finance probabilities_from_counts: explicit identity/reverse if used',
 'FinanceCase.analyze: q0-first indexing']

{'identity': {'counts': {'000': 162,
   '010': 148,
   '111': 108,
   '001': 154,
   '100': 106,
   '110': 67,
   '011': 201,
   '101': 78},
  'positions': [0, 1, 2],
  'origin': 'diagnostic identity interpretation',
  'applied_to_original': False},
 'reverse': {'counts': {'000': 162,
   '010': 148,
   '111': 108,
   '100': 154,
   '001': 106,
   '011': 67,
   '110': 201,
   '101': 78},
  'positions': [2, 1, 0],
  'origin': 'finance q0-last assumption, not service metadata',
  'applied_to_original': False},
 'ascending_physical_register': {'counts': {'000': 162,
   '001': 148,
   '111': 108,
   '100': 154,
   '010': 106,
   '011': 67,
   '101': 201,
   '110': 78},
  'positions': [2, 0, 1],
  'origin': 'HYPOTHESIS: raw labels sorted by physical index; mapping alone does not prove it',
  'applied_to_original': False}}

## 4. 三个反例及禁止提交保护

文档里的假 bit_order 不能传播成历史字段；本地接受 target_qubits 后仍映射到另一集合时，
不能称为锁定布局；同一组 counts 的 identity／reverse／物理排序解释必须保留来源、可逆且不覆盖原始结果。
下面仅用本地字典复现，全部是假设性软件反例，不是真机任务。


In [5]:
from quantumbridge.finance.client_contract import response_field_matrix, layout_pinning_verdict, trace_count_labels
from quantumbridge.compat.quafu import QuafuCloudBackend
from quantumbridge.providers import JobError
fake_documentation = {"bit_order": "c0_rightmost"}
historical = {"count": {"001": 200, "100": 824}}
assert response_field_matrix(historical, documentation=fake_documentation)["bit_order"]["status"] == "ABSENT"
assert layout_pinning_verdict([3,4,10,11], [10,11,4,5])["status"] == "ERROR"
display(trace_count_labels(historical, 3, ordered_physical_mapping=[11,4,5])["hypotheses"])
class NeverSubmit:
    def submit(self, *args, **kwargs):
        raise AssertionError("不应触及提交网关")
backend = QuafuCloudBackend(transport=NeverSubmit(), read_only=True)
try:
    backend.run(object())
except JobError as error:
    print(str(error))
else:
    raise AssertionError("禁止提交保护没有生效")


{'identity': {'counts': {'001': 200, '100': 824},
  'positions': [0, 1, 2],
  'origin': 'diagnostic identity interpretation',
  'applied_to_original': False},
 'reverse': {'counts': {'100': 200, '001': 824},
  'positions': [2, 1, 0],
  'origin': 'finance q0-last assumption, not service metadata',
  'applied_to_original': False},
 'ascending_physical_register': {'counts': {'100': 200, '010': 824},
  'positions': [2, 0, 1],
  'origin': 'HYPOTHESIS: raw labels sorted by physical index; mapping alone does not prove it',
  'applied_to_original': False}}

read-only cloud backend refuses submission


## 5. 决策和供应商待确认问题

当前不申请新增真机任务。先确认 counts 字符串排序、Task.result 是否转换标签、
历史 compiled measurement 的语义、布局固定保证、零任务／零收费 compile-only 契约和历史 calibration ID。
这些问题尚未向供应商发送，不能把草案当成其回复。若未来能够验证目标布局，再形成具体 QASM 和最小预算交 owner 决定。


In [6]:
display(load("layout_pinning_capability.json"))
display(load("compile_only_capability.json"))
proposal = load("next_experiment_approval_proposal.json")
display(proposal)
assert proposal["execute"] is False and proposal["additional_tasks_requested_now"] == 0
print("NO_NEW_SUBMISSION_RECOMMENDED；原始数值不修正，金融精度仍未通过验收。")


{'status': 'DOCUMENTED_UNVERIFIED',
 'decision': 'PHYSICAL_LAYOUT_PINNING_NOT_VERIFIED',
 'documented_parameter': 'options.target_qubits',
 'adapter_accepts_parameter': False,
 'server_guarantee_observed': False,
 'target_backend': 'Dongling',
 'sources': {'historical_quarkstudio': 'https://quarkstudio.readthedocs.io/en/latest/usage/code/transpiler/',
  'quafusqc_guide': 'https://quafu-sqc.readthedocs.io/en/latest/',
  'separate_quafu_sdk_0_2_2': 'https://pypi.org/project/quafu-sdk/0.2.2/'}}

{'status': 'UNSUPPORTED',
 'scope': 'current project adapter',
 'platform_status': 'DOCUMENTED_UNVERIFIED',
 'public_Task_methods': ['backend',
  'cancel',
  'delete',
  'query',
  'request',
  'result',
  'run',
  'status',
  'verify'],
 'trial_execution': False,
 'compile_false_means_free': False}

{'decision': 'NO_NEW_SUBMISSION_RECOMMENDED',
 'execute': False,
 'additional_tasks_requested_now': 0,
 'remaining_budget': 7,
 'owner_approval_required_for_future_plan': True,
 'supplier_questions_not_sent': ['Dongling raw-count label serialization order and registers',
  'whether Task.result transforms server labels',
  'historical compiled mapping semantics',
  'fixed physical layout guarantee and whether it is ordered or set-only',
  'pre-execution compile-only API with explicit zero-task/zero-charge guarantee',
  'historical calibration ID availability'],
 'conditional_minimal_plan': 'only after layout/label contracts are verified, review the D2 two-pattern plan with actual QASM and owner approval'}

NO_NEW_SUBMISSION_RECOMMENDED；原始数值不修正，金融精度仍未通过验收。


## 资料来源

- [QuarkStudio Transpiler 文档](https://quarkstudio.readthedocs.io/en/latest/usage/code/transpiler/)：历史任务与 target_qubits 示例。
- [Quafu-SQC 使用说明](https://quafu-sqc.readthedocs.io/en/latest/)：任务查询和返回字典。
- [另一个 quafu-sdk 发行包自述](https://pypi.org/project/quafu-sdk/0.2.2/)：不同的结果对象契约，未安装、未用于历史证据回填。

本附件不包含资料页面的 token 示例，不复制第三方实现，不代表全 SDK 功能验收完成。
